# Distributed Task Processing with TaskVine

This is a starter notebook for a Floability backpack. It demonstrates:
1. Connecting to a TaskVine manager
2. Defining a distributed worker function
3. Submitting and collecting tasks

**To customize this notebook:**
- Edit the `worker_function()` to implement your logic
- Modify the task parameters or task generation loop
- Add more worker functions or distribute more complex tasks

## Setup: Manager Connection

TaskVine requires a manager process. The manager name and ports come from environment variables set by Floability.

In [ ]:
import os
import ndcctools.taskvine as vine

# Get manager info from environment (set by Floability)
manager_name = os.environ.get('VINE_MANAGER_NAME')
manager_ports = os.environ.get('VINE_MANAGER_PORTS', '9123,9150')

print(f'Manager Name: {manager_name}')
print(f'Manager Ports: {manager_ports}')

# Connect to the manager
if manager_name and manager_ports:
    port_range = manager_ports.split(',')
    min_port = int(port_range[0])
    max_port = int(port_range[1])
    q = vine.Manager(port=min_port)
    print(f'Manager listening on port {q.port}')
else:
    print('Manager info not available. Ensure running via Floability.')

## Define Worker Function

The worker function will run on distributed worker nodes. In this example, it doubles a number and sleeps briefly.

**To customize:** Replace the function body with your processing logic.

In [ ]:
import time

def worker_function(value, sleep_time=1):
    """Simple worker function that doubles a number.
    
    Args:
        value: Input integer
        sleep_time: Time to sleep in seconds (simulates work)
    
    Returns:
        Dictionary with input and output
    """
    time.sleep(sleep_time)
    result = value * 2
    return {
        'input': value,
        'output': result,
    }

# Test the function locally
print('Worker function defined.')
print(f'Test: worker_function(5) = {worker_function(5, sleep_time=0)}')

## Submit Tasks to Workers

This cell submits tasks to the TaskVine workers. Each task will be processed by a worker node.

**To customize:** Modify the task generation loop to match your use case.

In [ ]:
if 'q' in locals():
    # Generate tasks: double numbers 1 through 10
    num_tasks = 10
    for i in range(1, num_tasks + 1):
        t = vine.PythonTask(
            worker_function,
            i,
            sleep_time=1,
        )
        q.submit(t)
    
    print(f'Submitted {num_tasks} tasks to the manager')
else:
    print('Manager not connected. Cannot submit tasks.')

## Collect Results

Wait for tasks to complete and collect results.

In [ ]:
if 'q' in locals():
    results = []
    total_submitted = q.submitted
    
    while len(results) < total_submitted:
        t = q.wait(5)
        if t:
            result = t.output
            results.append(result)
            print(f'Task completed: input={result["input"]} output={result["output"]}')
    
    print(f'\nAll {len(results)} task(s) completed!')
    print('\nResults:')
    for r in sorted(results, key=lambda x: x['input']):
        print(f'  {r["input"]} * 2 = {r["output"]}')
else:
    print('Manager not connected.')